# Running GACL (Graph Adversarial Contrastive Learning)

This notebook shows how to run the GACL rumor-detection model in `main.py`.

Reference: Tiening Sun et al., *Rumor Detection on Social Media with Graph Adversarial Contrastive Learning*, WWW '22.

**Before you start**, make sure the datasets and pre-trained word vectors have already been downloaded and placed in the correct folders (see `README.md`). The code expects this layout (default dataset is `Twitter16`):

```
3-GNN-Baselines/
├── data/
│   ├── twitter16/<event_id>/after_tweets.pkl
│   ├── twitter16/<event_id>/after_structure.pkl
│   ├── label_16.json
│   └── Twitter16_label_All.txt
├── bert_w2c/
│   └── T16/
│       ├── t16_mask_00/<event_id>.json
│       └── t16_mask_015/<event_id>.json
└── main.py
```


## 1. Install dependencies

GACL needs PyTorch, PyTorch Geometric (and `torch-scatter`), NumPy and tqdm. Install the versions that match your CUDA / PyTorch setup. The commands below are an example for CPU/CUDA — adjust the wheel URL to your own PyTorch and CUDA version (see https://pytorch-geometric.readthedocs.io for the right index URL).

In [ ]:
# Example install (uncomment and adapt to your environment):
# !pip install torch torchvision
# !pip install torch_geometric
# !pip install torch_scatter -f https://data.pyg.org/whl/torch-$(python -c "import torch;print(torch.__version__)").html
# !pip install numpy tqdm

## 2. Verify the environment and data layout

Run this check first — it confirms the imports work and that the expected data folders exist before launching a long training run.

In [ ]:
import os

# Make sure the working directory is the repository root (where main.py lives).
# If needed, change into it, e.g.:
# os.chdir('/path/to/3-GNN-Baselines')
print('Working dir:', os.getcwd())

import torch as th
import torch_geometric
import torch_scatter
import numpy as np
print('torch', th.__version__, '| pyg', torch_geometric.__version__, '| numpy', np.__version__)
print('CUDA available:', th.cuda.is_available())

for p in ['./data/twitter16', './data/label_16.json', './data/Twitter16_label_All.txt',
          './bert_w2c/T16/t16_mask_00', './bert_w2c/T16/t16_mask_015']:
    print(('OK  ' if os.path.exists(p) else 'MISSING '), p)

## 3. Run the full 5-fold training

`main.py` runs the complete pipeline at module level: it loads the 5-fold split and trains/evaluates on each fold, then prints averaged accuracy and per-class F1. The simplest way to run it from a notebook is to execute the script directly.

In [ ]:
# Option A: run as an external process (output streams to the notebook).
!python main.py

In [ ]:
# Option B: run inside the notebook kernel so you can inspect variables afterwards.
# %run main.py

## 4. Adjusting settings

The hyperparameters and dataset selection live at the bottom of `main.py` (the `main` section):

- `datasetname` — `'Twitter16'` by default. The code paths in `Process/dataset.py` are hard-coded to the Twitter16 files (`./data/twitter16/`, `./bert_w2c/T16/...`, `label_16.json`); switching to Twitter15/PHEME/Weibo requires also updating those paths and the matching tuning constants noted in the comments (e.g. `t`, `droprate`, `epsilon`).
- `lr`, `weight_decay`, `patience`, `n_epochs`, `batchsize` — standard training knobs.
- `device` — uses `cuda:0` when a GPU is available, otherwise CPU.

When the best validation score improves (after epoch 25), `EarlyStopping.save_checkpoint` saves the model weights to a file named `GACL_<datasetname>.m` in the working directory.